In [55]:
import pandas as pd
import os
import re
import numpy as np

In [56]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input/whirlpool")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output/whirlpool")

In [57]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    df = df.copy()

    df["sector"] = df[strategy_col].str.extract(r"TX:([A-Z]{3,6}):", expand=False)

    df.loc[df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False), "sector"] = "BASE"

    df["transformation_name"] = df[strategy_col].str.extract(r"TX:[A-Z]{3,6}:(\S+)", expand=False)

    base_mask = df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    # Elimina "_STRATEGY_XX" al final
    df["transformation_name"] = df["transformation_name"].str.replace(r"_STRATEGY_\w+$", "", regex=True)

    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [58]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "raw_emissions_libya_2022_whirlpool_data_raw.csv"))

emissions_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000293,2022,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000301,2023,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000309,2024,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000317,2025,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000326,2026,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE


In [59]:
print(emissions_df.primary_id.nunique())

43


In [60]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'LEP',
       'Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_LEP from LEP',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_DEMAND_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_HFCS_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_N2O_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_OTHER_FCS_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_PFCS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_CAPTURE_BIOGAS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_CATTLE_PIGS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_OTHER_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_POULTRY_STRATEGY_LEP from LEP',
       'Remove TX:LVST:DEC_ENTERIC_FERMENTATION_STRATEGY_LEP from LEP',
       'Remove TX:LVST:DEC_EX

In [61]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

44
42


In [62]:
filtered_emissions_df["strategy"].unique()

array(['LEP', 'Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_LEP from LEP',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_DEMAND_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_HFCS_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_N2O_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_OTHER_FCS_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_PFCS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_CAPTURE_BIOGAS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_CATTLE_PIGS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_OTHER_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_POULTRY_STRATEGY_LEP from LEP',
       'Remove TX:LVST:DEC_ENTERIC_FERMENTATION_STRATEGY_LEP from LEP',
       'Remove TX:LVST:DEC_EXPORTS_STRATEGY_LEP from LEP

In [63]:
filtered_emissions_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
1392,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000293,2022,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE
1393,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000301,2023,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE
1394,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000309,2024,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE
1395,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000317,2025,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE
1396,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000326,2026,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE


In [64]:
filtered_emissions_df.tail()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
59851,6045.0,116116.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.048468,2046,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_LEP from...,LBY,libya,SISEPUEDE
59852,6045.0,116116.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.046098,2047,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_LEP from...,LBY,libya,SISEPUEDE
59853,6045.0,116116.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.044004,2048,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_LEP from...,LBY,libya,SISEPUEDE
59854,6045.0,116116.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.042198,2049,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_LEP from...,LBY,libya,SISEPUEDE
59855,6045.0,116116.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.040689,2050,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_LEP from...,LBY,libya,SISEPUEDE


In [65]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['LEP', 'Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_LEP from LEP',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_DEMAND_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_HFCS_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_N2O_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_OTHER_FCS_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_PFCS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_CAPTURE_BIOGAS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_CATTLE_PIGS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_OTHER_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_POULTRY_STRATEGY_LEP from LEP',
       'Remove TX:LVST:DEC_ENTERIC_FERMENTATION_STRATEGY_LEP from LEP',
       'Remove TX:LVST:DEC_EXPORTS_STRATEGY_LEP from LEP

In [66]:
tornado_emissions_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
1392,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000293,2022,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE
1393,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000301,2023,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE
1394,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000309,2024,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE
1395,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000317,2025,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE
1396,6004.0,75075.0,AG - Crops:CH4,Agriculture,AG - Crops,0.000326,2026,CH4,0.0,0.0,LEP,LBY,libya,SISEPUEDE


In [67]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,value
0,6004.0,75075.0,LEP,2169.285272
1,6005.0,76076.0,Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP,2025.573790
2,6006.0,77077.0,Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_...,3029.096779
3,6007.0,78078.0,Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP,2384.612848
4,6008.0,79079.0,Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP,2188.257038


In [68]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,value
37,6041.0,112112.0,Remove TX:WASO:INC_CAPTURE_BIOGAS_STRATEGY_LEP...,2174.816103
38,6042.0,113113.0,Remove TX:WASO:INC_ENERGY_FROM_BIOGAS_STRATEGY...,2169.142869
39,6043.0,114114.0,Remove TX:WASO:INC_ENERGY_FROM_INCINERATION_ST...,2170.415844
40,6044.0,115115.0,Remove TX:WASO:INC_LANDFILLING_STRATEGY_LEP fr...,2176.909379
41,6045.0,116116.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_LEP from...,2175.932515


In [69]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

42
42


In [70]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total column by setting it to the strategy_id == 0 value
base_emission_total = (tornado_emissions_agg_df.loc[tornado_emissions_agg_df['strategy_id'] == 6004, 'emission_total'].values[0]) 
tornado_emissions_agg_df['base_emission_total'] = base_emission_total 

base_emission_total

np.float64(2169.2852718528948)

In [71]:
# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] =  tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']

tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_diff'].round(1)

tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff
0,6004.0,75075.0,LEP,2169.285272,2169.285272,0.0
1,6005.0,76076.0,Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP,2025.573790,2169.285272,-143.7
2,6006.0,77077.0,Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_...,3029.096779,2169.285272,859.8
3,6007.0,78078.0,Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP,2384.612848,2169.285272,215.3
4,6008.0,79079.0,Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP,2188.257038,2169.285272,19.0


In [72]:
tornado_emissions_agg_extended_df = add_sector_and_transformation_fields(tornado_emissions_agg_df)
tornado_emissions_agg_extended_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,6004.0,75075.0,LEP,2169.285272,2169.285272,0.0,NaN,
1,6005.0,76076.0,Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP,2025.573790,2169.285272,-143.7,ENFU,ADJ_EXPORTS
2,6006.0,77077.0,Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_...,3029.096779,2169.285272,859.8,ENTC,TARGET_RENEWABLE_ELEC
3,6007.0,78078.0,Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP,2384.612848,2169.285272,215.3,FGTV,DEC_LEAKS
4,6008.0,79079.0,Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP,2188.257038,2169.285272,19.0,FGTV,INC_FLARE


In [73]:
tornado_emissions_agg_extended_df.to_clipboard(index=False)

## Load and process CB data

In [74]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "cba_results_ssp_modeling_whirlpool.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value
0,PFLO:LEP,0.0,libya,7.0,pop_unimproved_rural,982602.524082,982602.524082,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
1,PFLO:LEP,0.0,libya,8.0,pop_unimproved_rural,979649.346555,979649.346555,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
2,PFLO:LEP,0.0,libya,9.0,pop_unimproved_rural,976696.169029,976696.169029,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
3,PFLO:LEP,0.0,libya,10.0,pop_unimproved_rural,973742.991502,973742.991502,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0
4,PFLO:LEP,0.0,libya,11.0,pop_unimproved_rural,968072.540539,968072.540539,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0


In [75]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

# Split 'variable' into components: name, sector, cb_type, item_1, item_2
# (Assumes exactly 5 colon-separated parts; if there are more colons inside the last field,
# they will be kept in item_2 thanks to n=4)
cb_chars = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
cb_chars.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
cb_data = pd.concat([cb_data, cb_chars], axis=1)
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2
0,PFLO:LEP,0.0,libya,7.0,pop_unimproved_rural,982602.524082,982602.524082,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
1,PFLO:LEP,0.0,libya,8.0,pop_unimproved_rural,979649.346555,979649.346555,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
2,PFLO:LEP,0.0,libya,9.0,pop_unimproved_rural,976696.169029,976696.169029,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
3,PFLO:LEP,0.0,libya,10.0,pop_unimproved_rural,973742.991502,973742.991502,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural
4,PFLO:LEP,0.0,libya,11.0,pop_unimproved_rural,968072.540539,968072.540539,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural


In [76]:
# Scale value from USD to billions (divide by 1e9)
if "value" in cb_data.columns:
    cb_data["value"] = cb_data["value"] / 1e9

# --- Remove "shifted" entries ---
# # Remove rows where item_2 contains "shifted"
# cb_data = cb_data[~cb_data["item_2"].astype(str).str.contains("shifted", na=False)]

# # Remove any remaining rows where variable contains "shifted2"
# cb_data = cb_data[~cb_data["variable"].astype(str).str.contains("shifted2", na=False)]

# --- Add Year column (Year = time_period + 2015) ---
cb_data["Year"] = cb_data["time_period"] + 2015

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year
0,PFLO:LEP,0.0,libya,7.0,pop_unimproved_rural,982602.524082,982602.524082,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0
1,PFLO:LEP,0.0,libya,8.0,pop_unimproved_rural,979649.346555,979649.346555,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0
2,PFLO:LEP,0.0,libya,9.0,pop_unimproved_rural,976696.169029,976696.169029,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0
3,PFLO:LEP,0.0,libya,10.0,pop_unimproved_rural,973742.991502,973742.991502,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0
4,PFLO:LEP,0.0,libya,11.0,pop_unimproved_rural,968072.540539,968072.540539,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0


In [77]:
# Load attribute strategy
attribute_strategy_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "ATTRIBUTE_STRATEGY.csv"))
attribute_strategy_df = attribute_strategy_df[["strategy_id", "strategy_code"]]
attribute_strategy_df.head()

,strategy_id,strategy_code
0,0,BASE
1,3006,FGTV:DEC_LEAKS
2,6004,PFLO:LEP
3,6005,WHIRLPOOL:TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP
4,6006,WHIRLPOOL:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATE...


In [78]:
attribute_strategy_df.strategy_id.unique()

array([   0, 3006, 6004, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6012,
       6013, 6014, 6015, 6016, 6017, 6018, 6019, 6020, 6021, 6022, 6023,
       6024, 6025, 6026, 6027, 6028, 6029, 6030, 6031, 6032, 6033, 6034,
       6035, 6036, 6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045])

In [79]:
attribute_strategy_df.strategy_code.unique()

array(['BASE', 'FGTV:DEC_LEAKS', 'PFLO:LEP',
       'WHIRLPOOL:TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP',
       'WHIRLPOOL:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_LEP',
       'WHIRLPOOL:TX:FGTV:DEC_LEAKS_STRATEGY_LEP',
       'WHIRLPOOL:TX:FGTV:INC_FLARE_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_CLINKER_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_DEMAND_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_HFCS_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_N2O_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_OTHER_FCS_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_PFCS_STRATEGY_LEP',
       'WHIRLPOOL:TX:LSMM:INC_CAPTURE_BIOGAS_STRATEGY_LEP',
       'WHIRLPOOL:TX:LSMM:INC_MANAGEMENT_CATTLE_PIGS_STRATEGY_LEP',
       'WHIRLPOOL:TX:LSMM:INC_MANAGEMENT_OTHER_STRATEGY_LEP',
       'WHIRLPOOL:TX:LSMM:INC_MANAGEMENT_POULTRY_STRATEGY_LEP',
       'WHIRLPOOL:TX:LVST:DEC_ENTERIC_FERMENTATION_STRATEGY_LEP',
       'WHIRLPOOL:TX:LVST:DEC_EXPORTS_STRATEGY_LEP',
       'WHIRLPOOL:TX:LVST:INC_PRODUCTIVITY_STRATEGY_LEP

In [80]:
# Merge with cb_data on strategy_code
cb_data = cb_data.merge(attribute_strategy_df, on="strategy_code", how="left")
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:LEP,0.0,libya,7.0,pop_unimproved_rural,982602.524082,982602.524082,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6004
1,PFLO:LEP,0.0,libya,8.0,pop_unimproved_rural,979649.346555,979649.346555,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6004
2,PFLO:LEP,0.0,libya,9.0,pop_unimproved_rural,976696.169029,976696.169029,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6004
3,PFLO:LEP,0.0,libya,10.0,pop_unimproved_rural,973742.991502,973742.991502,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6004
4,PFLO:LEP,0.0,libya,11.0,pop_unimproved_rural,968072.540539,968072.540539,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6004


In [81]:
cb_data.strategy_code.unique()

array(['PFLO:LEP', 'WHIRLPOOL:TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP',
       'WHIRLPOOL:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_LEP',
       'WHIRLPOOL:TX:FGTV:DEC_LEAKS_STRATEGY_LEP',
       'WHIRLPOOL:TX:FGTV:INC_FLARE_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_CLINKER_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_DEMAND_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_HFCS_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_N2O_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_OTHER_FCS_STRATEGY_LEP',
       'WHIRLPOOL:TX:IPPU:DEC_PFCS_STRATEGY_LEP',
       'WHIRLPOOL:TX:LSMM:INC_CAPTURE_BIOGAS_STRATEGY_LEP',
       'WHIRLPOOL:TX:LSMM:INC_MANAGEMENT_CATTLE_PIGS_STRATEGY_LEP',
       'WHIRLPOOL:TX:LSMM:INC_MANAGEMENT_OTHER_STRATEGY_LEP',
       'WHIRLPOOL:TX:LSMM:INC_MANAGEMENT_POULTRY_STRATEGY_LEP',
       'WHIRLPOOL:TX:LVST:DEC_ENTERIC_FERMENTATION_STRATEGY_LEP',
       'WHIRLPOOL:TX:LVST:DEC_EXPORTS_STRATEGY_LEP',
       'WHIRLPOOL:TX:LVST:INC_PRODUCTIVITY_STRATEGY_LEP',
       'WHIRLPOOL:TX:PFLO:INC_

In [82]:
cb_data.strategy_id.unique()

array([6004, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6012, 6013, 6014,
       6015, 6016, 6017, 6018, 6019, 6020, 6021, 6022, 6023, 6024, 6025,
       6026, 6027, 6028, 6029, 6030, 6031, 6032, 6033, 6034, 6035, 6036,
       6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045])

In [83]:
# check for nans in strategy_id
cb_data[cb_data['strategy_id'].isna()]['strategy_code'].unique()

array([], dtype=object)

In [84]:
cb_data["sector"].unique()

array(['wali', 'entc', 'trns', 'lndu', 'waso', 'trww', 'lvst', 'agrc',
       'ccsq', 'inen', 'scoe', 'ippu', 'soil', 'lsmm', 'fgtv', 'pflo'],
      dtype=object)

In [85]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:LEP,0.0,libya,7.0,pop_unimproved_rural,982602.524082,982602.524082,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6004
1,PFLO:LEP,0.0,libya,8.0,pop_unimproved_rural,979649.346555,979649.346555,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6004
2,PFLO:LEP,0.0,libya,9.0,pop_unimproved_rural,976696.169029,976696.169029,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6004
3,PFLO:LEP,0.0,libya,10.0,pop_unimproved_rural,973742.991502,973742.991502,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6004
4,PFLO:LEP,0.0,libya,11.0,pop_unimproved_rural,968072.540539,968072.540539,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6004


In [86]:
# filter sectors
# target_sectors = ["wali", "trww", "waso", "soil", "ippu", "lvst", "agrc", "lndu", "lsmm"]
# cb_data = cb_data[cb_data["sector"].isin(target_sectors)].copy()
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:LEP,0.0,libya,7.0,pop_unimproved_rural,982602.524082,982602.524082,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6004
1,PFLO:LEP,0.0,libya,8.0,pop_unimproved_rural,979649.346555,979649.346555,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6004
2,PFLO:LEP,0.0,libya,9.0,pop_unimproved_rural,976696.169029,976696.169029,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6004
3,PFLO:LEP,0.0,libya,10.0,pop_unimproved_rural,973742.991502,973742.991502,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6004
4,PFLO:LEP,0.0,libya,11.0,pop_unimproved_rural,968072.540539,968072.540539,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6004


In [87]:
# aggregate sum(value) grouped by strategy_id and cb_type
cb_data = (
    cb_data.groupby(["strategy_id", "cb_type"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head(20)

,strategy_id,cb_type,cumulative
0,6004,air_pollution,3.872383e+01
1,6004,congestion,2.114609e+01
2,6004,consumer_savings,2.071405e+01
3,6004,crop_value,3.352761e-15
4,6004,ecosystem_services,0.000000e+00
5,6004,env_pollution,1.220097e+01
6,6004,fuel_cost,8.228626e+00
7,6004,human_health,3.337513e+01
8,6004,ippu_value,4.045291e+01
9,6004,land_pollution,0.000000e+00


In [88]:
# unique cb_data types
cb_cats = cb_data["cb_type"].unique().tolist()

# long -> wide (R dcast equivalent)
wide_cb = (
    cb_data.pivot(index="strategy_id", columns="cb_type", values="cumulative")
      .reset_index()
)

# optional: remove column name from pivot for nicer printing
wide_cb.columns.name = None
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,6004,38.723831,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,0.0,3.902153,32.125659,90.074321,216.465775,-138.671358,1.439357,14.0828
1,6005,39.553402,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,0.0,3.902153,32.125659,90.364856,216.465775,116.569186,1.439357,14.0828
2,6006,37.012647,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,0.0,3.902153,32.125659,89.999716,216.465775,-63.206515,1.439357,14.0828
3,6007,38.723831,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,0.0,3.902153,32.125659,90.074324,216.465775,-136.603174,1.439357,14.0828
4,6008,38.723831,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,0.0,3.902153,32.125659,90.074321,216.465775,-138.571294,1.439357,14.0828


In [89]:
cb_cats

['air_pollution',
 'congestion',
 'consumer_savings',
 'crop_value',
 'ecosystem_services',
 'env_pollution',
 'fuel_cost',
 'human_health',
 'ippu_value',
 'land_pollution',
 'lvst_value',
 'road_safety',
 'sector_specific',
 'system_cost',
 'technical_cost',
 'technical_savings',
 'water_pollution']

In [90]:
# --- 1) net_benefit = rowSums over all cb categories ---
wide_cb["net_benefit"] = wide_cb[cb_cats].sum(axis=1, skipna=True)

# --- 2) additional_benefits = rowSums excluding "technical_cost" ---
benefit_cols = [c for c in cb_cats if c != "technical_cost"]
wide_cb["additional_benefits"] = wide_cb[benefit_cols].sum(axis=1, skipna=True)

# --- 3) total_transformation_costs = rowSums over specific cols ---
cost_cols = ["technical_cost", "technical_savings", "fuel_cost"]

# (safe version: only use cols that exist in the df)
cost_cols = [c for c in cost_cols if c in wide_cb.columns]

wide_cb["total_transformation_costs"] = wide_cb[cost_cols].sum(axis=1, skipna=True)
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6004,38.723831,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,...,3.902153,32.125659,90.074321,216.465775,-138.671358,1.439357,14.0828,394.260318,532.931677,-129.003375
1,6005,39.553402,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,...,3.902153,32.125659,90.364856,216.465775,116.569186,1.439357,14.0828,650.620969,534.051783,126.237170
2,6006,37.012647,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,...,3.902153,32.125659,89.999716,216.465775,-63.206515,1.439357,14.0828,467.939372,531.145887,-53.538532
3,6007,38.723831,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,...,3.902153,32.125659,90.074324,216.465775,-136.603174,1.439357,14.0828,396.328505,532.931679,-126.935191
4,6008,38.723831,21.146089,20.714053,3.352761e-15,0.0,12.200971,8.228626,33.375129,40.452911,...,3.902153,32.125659,90.074321,216.465775,-138.571294,1.439357,14.0828,394.360383,532.931677,-128.903311


## Merge emissions and cb data and save

In [91]:
tornado_emissions_agg_extended_df[tornado_emissions_agg_extended_df.strategy == "Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP"]

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
1,6005.0,76076.0,Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP,2025.57379,2169.285272,-143.7,ENFU,ADJ_EXPORTS


In [92]:
wide_cb.strategy_id.unique()

array([6004, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6012, 6013, 6014,
       6015, 6016, 6017, 6018, 6019, 6020, 6021, 6022, 6023, 6024, 6025,
       6026, 6027, 6028, 6029, 6030, 6031, 6032, 6033, 6034, 6035, 6036,
       6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045])

In [93]:
print(wide_cb.shape)
print(tornado_emissions_agg_extended_df.shape)

(42, 21)
(42, 8)


In [94]:
df_merged = pd.merge(
    tornado_emissions_agg_extended_df,
    wide_cb,
    on="strategy_id",
    how="inner"
)

df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6004.0,75075.0,LEP,2169.285272,2169.285272,0.0,NaN,,38.723831,21.146089,...,3.902153,32.125659,90.074321,216.465775,-138.671358,1.439357,14.0828,394.260318,532.931677,-129.003375
1,6005.0,76076.0,Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP,2025.573790,2169.285272,-143.7,ENFU,ADJ_EXPORTS,39.553402,21.146089,...,3.902153,32.125659,90.364856,216.465775,116.569186,1.439357,14.0828,650.620969,534.051783,126.237170
2,6006.0,77077.0,Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_...,3029.096779,2169.285272,859.8,ENTC,TARGET_RENEWABLE_ELEC,37.012647,21.146089,...,3.902153,32.125659,89.999716,216.465775,-63.206515,1.439357,14.0828,467.939372,531.145887,-53.538532
3,6007.0,78078.0,Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP,2384.612848,2169.285272,215.3,FGTV,DEC_LEAKS,38.723831,21.146089,...,3.902153,32.125659,90.074324,216.465775,-136.603174,1.439357,14.0828,396.328505,532.931679,-126.935191
4,6008.0,79079.0,Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP,2188.257038,2169.285272,19.0,FGTV,INC_FLARE,38.723831,21.146089,...,3.902153,32.125659,90.074321,216.465775,-138.571294,1.439357,14.0828,394.360383,532.931677,-128.903311


In [95]:
df_merged = df_merged[df_merged['strategy'] != 'LEP']

df_merged.head()


,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
1,6005.0,76076.0,Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP,2025.573790,2169.285272,-143.7,ENFU,ADJ_EXPORTS,39.553402,21.146089,...,3.902153,32.125659,90.364856,216.465775,116.569186,1.439357,14.0828,650.620969,534.051783,126.237170
2,6006.0,77077.0,Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_...,3029.096779,2169.285272,859.8,ENTC,TARGET_RENEWABLE_ELEC,37.012647,21.146089,...,3.902153,32.125659,89.999716,216.465775,-63.206515,1.439357,14.0828,467.939372,531.145887,-53.538532
3,6007.0,78078.0,Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP,2384.612848,2169.285272,215.3,FGTV,DEC_LEAKS,38.723831,21.146089,...,3.902153,32.125659,90.074324,216.465775,-136.603174,1.439357,14.0828,396.328505,532.931679,-126.935191
4,6008.0,79079.0,Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP,2188.257038,2169.285272,19.0,FGTV,INC_FLARE,38.723831,21.146089,...,3.902153,32.125659,90.074321,216.465775,-138.571294,1.439357,14.0828,394.360383,532.931677,-128.903311
5,6009.0,80080.0,Remove TX:IPPU:DEC_CLINKER_STRATEGY_LEP from LEP,2169.285272,2169.285272,0.0,IPPU,DEC_CLINKER,38.723831,21.146089,...,3.902153,32.125659,90.074321,216.465775,-132.113210,1.439357,14.0828,400.818467,532.931677,-122.445226


In [96]:
print(df_merged.shape)

(41, 28)


### Below we have some hardcoded fixed exclusive of this study case to replace incorrect tranformation names

In [97]:
df_merged.columns

Index(['strategy_id', 'primary_id', 'strategy', 'emission_total',
       'base_emission_total', 'emission_diff', 'sector', 'transformation_name',
       'air_pollution', 'congestion', 'consumer_savings', 'crop_value',
       'ecosystem_services', 'env_pollution', 'fuel_cost', 'human_health',
       'ippu_value', 'land_pollution', 'lvst_value', 'road_safety',
       'sector_specific', 'system_cost', 'technical_cost', 'technical_savings',
       'water_pollution', 'net_benefit', 'additional_benefits',
       'total_transformation_costs'],
      dtype='object')

In [98]:
# Get the base technical cost from wide_cb (strategy_id 6004)
base_technical_cost = (wide_cb[wide_cb['strategy_id'] == 6004]['technical_cost'].values[0])*-1
base_technical_cost

np.float64(138.6713584749956)

In [99]:
# multiply technical_cost by -1 to get positive costs
df_merged['technical_cost'] = df_merged['technical_cost'] * -1

df_merged['technical_cost'] = base_technical_cost - df_merged['technical_cost'] 

# create marginal total abatement cost column
df_merged['marginal_total_abatement_cost_(USD/tCO2e)'] = (df_merged['technical_cost'] / df_merged['emission_diff'])*1000

# If technical_cost is positive then marginal_total_abatement_cost should be positive too.
df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs() * np.sign(df_merged["technical_cost"])


In [100]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e)
1,6005.0,76076.0,Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP,2025.573790,2169.285272,-143.7,ENFU,ADJ_EXPORTS,39.553402,21.146089,...,32.125659,90.364856,216.465775,255.240544,1.439357,14.0828,650.620969,534.051783,126.237170,1776.204206
2,6006.0,77077.0,Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_...,3029.096779,2169.285272,859.8,ENTC,TARGET_RENEWABLE_ELEC,37.012647,21.146089,...,32.125659,89.999716,216.465775,75.464843,1.439357,14.0828,467.939372,531.145887,-53.538532,87.770229
3,6007.0,78078.0,Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP,2384.612848,2169.285272,215.3,FGTV,DEC_LEAKS,38.723831,21.146089,...,32.125659,90.074324,216.465775,2.068184,1.439357,14.0828,396.328505,532.931679,-126.935191,9.606058
4,6008.0,79079.0,Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP,2188.257038,2169.285272,19.0,FGTV,INC_FLARE,38.723831,21.146089,...,32.125659,90.074321,216.465775,0.100064,1.439357,14.0828,394.360383,532.931677,-128.903311,5.266538
5,6009.0,80080.0,Remove TX:IPPU:DEC_CLINKER_STRATEGY_LEP from LEP,2169.285272,2169.285272,0.0,IPPU,DEC_CLINKER,38.723831,21.146089,...,32.125659,90.074321,216.465775,6.558149,1.439357,14.0828,400.818467,532.931677,-122.445226,inf


In [101]:
df_merged["strategy"].unique()

array(['Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_LEP from LEP',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_DEMAND_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_HFCS_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_N2O_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_OTHER_FCS_STRATEGY_LEP from LEP',
       'Remove TX:IPPU:DEC_PFCS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_CAPTURE_BIOGAS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_CATTLE_PIGS_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_OTHER_STRATEGY_LEP from LEP',
       'Remove TX:LSMM:INC_MANAGEMENT_POULTRY_STRATEGY_LEP from LEP',
       'Remove TX:LVST:DEC_ENTERIC_FERMENTATION_STRATEGY_LEP from LEP',
       'Remove TX:LVST:DEC_EXPORTS_STRATEGY_LEP from LEP',
    

In [102]:
df_merged.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_whirlpool.csv"), index=False)

### Create a QA version

In [103]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e)
1,6005.0,76076.0,Remove TX:ENFU:ADJ_EXPORTS_STRATEGY_LEP from LEP,2025.573790,2169.285272,-143.7,ENFU,ADJ_EXPORTS,39.553402,21.146089,...,32.125659,90.364856,216.465775,255.240544,1.439357,14.0828,650.620969,534.051783,126.237170,1776.204206
2,6006.0,77077.0,Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_...,3029.096779,2169.285272,859.8,ENTC,TARGET_RENEWABLE_ELEC,37.012647,21.146089,...,32.125659,89.999716,216.465775,75.464843,1.439357,14.0828,467.939372,531.145887,-53.538532,87.770229
3,6007.0,78078.0,Remove TX:FGTV:DEC_LEAKS_STRATEGY_LEP from LEP,2384.612848,2169.285272,215.3,FGTV,DEC_LEAKS,38.723831,21.146089,...,32.125659,90.074324,216.465775,2.068184,1.439357,14.0828,396.328505,532.931679,-126.935191,9.606058
4,6008.0,79079.0,Remove TX:FGTV:INC_FLARE_STRATEGY_LEP from LEP,2188.257038,2169.285272,19.0,FGTV,INC_FLARE,38.723831,21.146089,...,32.125659,90.074321,216.465775,0.100064,1.439357,14.0828,394.360383,532.931677,-128.903311,5.266538
5,6009.0,80080.0,Remove TX:IPPU:DEC_CLINKER_STRATEGY_LEP from LEP,2169.285272,2169.285272,0.0,IPPU,DEC_CLINKER,38.723831,21.146089,...,32.125659,90.074321,216.465775,6.558149,1.439357,14.0828,400.818467,532.931677,-122.445226,inf


In [104]:
df_merged.sector.unique()

array(['ENFU', 'ENTC', 'FGTV', 'IPPU', 'LSMM', 'LVST', 'PFLO', 'SCOE',
       'TRDE', 'TRNS', 'TRWW', 'WALI', 'WASO'], dtype=object)

In [105]:
relevant_fields = [
    "transformation_name",
    "sector",
    "base_emission_total",
    "emission_total",
    "emission_diff",
    "technical_cost",
    "marginal_total_abatement_cost_(USD/tCO2e)"
]

# keep only relevant fields
df_merged_filtered = df_merged[relevant_fields]
df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,ADJ_EXPORTS,ENFU,2169.285272,2025.573790,-143.7,255.240544,1776.204206
2,TARGET_RENEWABLE_ELEC,ENTC,2169.285272,3029.096779,859.8,75.464843,87.770229
3,DEC_LEAKS,FGTV,2169.285272,2384.612848,215.3,2.068184,9.606058
4,INC_FLARE,FGTV,2169.285272,2188.257038,19.0,0.100064,5.266538
5,DEC_CLINKER,IPPU,2169.285272,2169.285272,0.0,6.558149,inf


In [106]:
df_merged_filtered

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,ADJ_EXPORTS,ENFU,2169.285272,2025.573790,-143.7,255.240544,1.776204e+03
2,TARGET_RENEWABLE_ELEC,ENTC,2169.285272,3029.096779,859.8,75.464843,8.777023e+01
3,DEC_LEAKS,FGTV,2169.285272,2384.612848,215.3,2.068184,9.606058e+00
4,INC_FLARE,FGTV,2169.285272,2188.257038,19.0,0.100064,5.266538e+00
5,DEC_CLINKER,IPPU,2169.285272,2169.285272,0.0,6.558149,inf
6,DEC_DEMAND,IPPU,2169.285272,2227.636453,58.4,-17.291545,-2.960881e+02
7,DEC_HFCS,IPPU,2169.285272,2215.826585,46.5,0.000000,0.000000e+00
8,DEC_N2O,IPPU,2169.285272,2175.478248,6.2,0.123860,1.997734e+01
9,DEC_OTHER_FCS,IPPU,2169.285272,2169.285272,0.0,0.000000,NaN
10,DEC_PFCS,IPPU,2169.285272,2169.285272,0.0,0.000000,NaN


In [107]:
df_merged_filtered.to_clipboard(index=False)

In [108]:
df_merged_filtered.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_for_QA_whirlpool.csv"), index=False)